# 142 — Semantic Caching

**What this workbook teaches:**
- How embedding-based caching differs from exact-match caching
- How cosine similarity works as a "semantic closeness" score
- How the `SemanticCache` class stores and retrieves responses
- How threshold choice (e.g. 0.85 vs 0.95) changes hit/miss behaviour

---

**Two sections:**
- **Part A** — Pure Python, no API key required. Walk through the mechanics with fake vectors.
- **Part B** — Live OpenAI calls. Requires `OPENAI_API_KEY` in your environment.

In [ ]:
%pip install -q openai python-dotenv langgraph

---
## Part A — Concepts (no API key needed)

### What is semantic caching?

**Exact-match caching** stores responses keyed by the exact query string. It only hits when the user types the identical question.

**Semantic caching** stores responses keyed by an *embedding* of the query — a vector of floats that captures meaning. A new query hits the cache if its embedding is close enough (above a similarity threshold) to a cached embedding, even if the wording differs.

**Workflow for each incoming query:**
1. Embed the query → `query_vec` (a list of floats)
2. Compare `query_vec` to every cached embedding using cosine similarity
3. If the best match ≥ threshold → **cache hit**, return stored response (no LLM call)
4. If no match ≥ threshold → **cache miss**, call LLM, store result

**Threshold tuning:**
- Higher threshold (e.g. 0.95) → fewer hits, but responses are very close matches
- Lower threshold (e.g. 0.80) → more hits, but may serve semantically adjacent but imprecise answers
- Default in this example: **0.92**

In [ ]:
# Part A — Cell 2: Cosine similarity with fake vectors
# This is the exact cosine_sim() from src/tools.py.

import math

def cosine_sim(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))
    return dot / (norm_a * norm_b + 1e-10)


# Identical vectors → similarity = 1.0
v1 = [0.8, 0.1, 0.5, 0.3]
v2 = [0.8, 0.1, 0.5, 0.3]
print(f"Identical vectors:                 {cosine_sim(v1, v2):.4f}")

# Similar vectors (small perturbation) → high similarity
v3 = [0.81, 0.09, 0.51, 0.29]
print(f"Similar vectors (minor tweak):     {cosine_sim(v1, v3):.4f}")

# Different topic vectors → low similarity
v4 = [0.1, 0.9, 0.0, 0.8]
print(f"Different topic vectors:           {cosine_sim(v1, v4):.4f}")

# Orthogonal vectors → similarity = 0.0
v5 = [0.0, 0.0, 1.0, 0.0]
v6 = [1.0, 0.0, 0.0, 0.0]
print(f"Orthogonal vectors:                {cosine_sim(v5, v6):.4f}")

print()
print("Interpretation: values close to 1.0 = same meaning, close to 0.0 = unrelated.")

In [ ]:
# Part A — Cell 3: Walk through SemanticCache with mock embeddings
# Uses the same class from src/tools.py but with hand-crafted vectors.

from dataclasses import dataclass, field

@dataclass
class SemanticCache:
    threshold: float = 0.92
    _entries: list = field(default_factory=list)
    hits: int = 0
    misses: int = 0

    def get(self, query_vec: list[float]) -> str | None:
        for vec, response in self._entries:
            if cosine_sim(query_vec, vec) >= self.threshold:
                self.hits += 1
                return response
        self.misses += 1
        return None

    def set(self, query_vec: list[float], response: str) -> None:
        self._entries.append((query_vec, response))

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": round(self.hits / total, 3) if total else 0,
        }


# Mock embeddings for 3 semantic clusters
# "machine learning" cluster
ml_vec    = [0.9, 0.1, 0.0, 0.1]
ml_vec2   = [0.89, 0.11, 0.01, 0.09]   # near-duplicate — should hit

# "deep learning" cluster  
dl_vec    = [0.7, 0.6, 0.1, 0.0]
dl_vec2   = [0.71, 0.59, 0.09, 0.01]   # near-duplicate — should hit

# "databases" — unrelated
db_vec    = [0.0, 0.0, 0.9, 0.8]

cache = SemanticCache(threshold=0.92)

queries = [
    (ml_vec,  "What is machine learning?",                 "ML is a field of AI that enables computers to learn from data."),
    (ml_vec2, "Can you explain machine learning to me?",   None),   # expect cache hit
    (dl_vec,  "What is deep learning?",                    "Deep learning uses neural networks with many layers."),
    (dl_vec2, "How does deep learning work?",              None),   # expect cache hit
    (db_vec,  "What is a relational database?",            None),   # expect miss
]

print(f"{'Source':<12} {'Query':<45} {'Response (truncated)'}")
print("-" * 90)
for vec, query, stored_response in queries:
    cached = cache.get(vec)
    if cached:
        source = "CACHE HIT"
        answer = cached
    else:
        # Simulate LLM call
        answer = stored_response or f"[LLM answer for: {query[:30]}...]"
        cache.set(vec, answer)
        source = "LLM CALL"
    print(f"{source:<12} {query:<45} {answer[:40]}")

print()
print("Stats:", cache.stats())

In [ ]:
# Part A — Cell 4: Threshold sensitivity demo
# Same mock queries run against threshold=0.85 vs threshold=0.95

def run_demo(threshold: float, query_vecs: list[tuple]) -> dict:
    """Run a cache simulation and return stats."""
    cache = SemanticCache(threshold=threshold)
    results = []
    for i, (query, vec, is_first_in_cluster) in enumerate(query_vecs):
        cached = cache.get(vec)
        if cached:
            results.append((query, "HIT"))
        else:
            cache.set(vec, f"Answer {i}")
            results.append((query, "MISS"))
    return {"stats": cache.stats(), "results": results}


# 6 queries: 3 semantic clusters, each with a near-duplicate
# Similarity within cluster ~0.997; across clusters ~0.3
query_vecs = [
    ("What is machine learning?",           [0.900, 0.100, 0.000, 0.100], True),
    ("Explain machine learning",            [0.895, 0.102, 0.002, 0.098], False),  # sim ~0.9997
    ("What is deep learning?",              [0.700, 0.600, 0.100, 0.000], True),
    ("How does deep learning work?",        [0.698, 0.601, 0.099, 0.001], False),  # sim ~0.9999
    ("What is a relational database?",      [0.000, 0.050, 0.900, 0.800], True),
    ("Explain relational databases",        [0.001, 0.048, 0.901, 0.799], False),  # sim ~0.9999
]

for threshold in [0.85, 0.92, 0.99]:
    out = run_demo(threshold, query_vecs)
    hits   = [q for q, r in out["results"] if r == "HIT"]
    misses = [q for q, r in out["results"] if r == "MISS"]
    print(f"\nThreshold = {threshold}  →  {out['stats']}")
    print(f"  Hits:   {hits}")
    print(f"  Misses: {misses}")

---
## Part B — Live OpenAI calls

**Requires:** `OPENAI_API_KEY` in your `.env` file or environment.

Runs 5 queries through the LangGraph workflow. Near-duplicate queries will hit the semantic cache instead of calling the LLM, and you'll see the hit rate and LLM call savings.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set. "
        "Add it to your .env file or export it before running Part B."
    )
print("OPENAI_API_KEY loaded.")

In [ ]:
# Part B — Run the semantic caching workflow with 5 queries
import sys, os, json
sys.path.insert(0, os.path.join(os.getcwd(), "."))

from openai import OpenAI
from src.workflow import create_workflow

QUERIES = [
    "What is machine learning?",
    "Can you explain machine learning to me?",   # near-duplicate → expect cache hit
    "What is deep learning?",
    "How does deep learning work?",              # near-duplicate → expect cache hit
    "What is a relational database?",            # new topic → expect miss
]

client = OpenAI(api_key=OPENAI_API_KEY)
workflow = create_workflow()
config = {"configurable": {"client": client}}

result = workflow.invoke({
    "queries": QUERIES,
    "threshold": 0.92,
    "responses": [],
    "cache_stats": {},
}, config=config)

print("=== Semantic Cache Demo ===\n")
llm_calls = 0
for r in result["responses"]:
    label = "[CACHE HIT]" if r["source"] == "cache" else f"[LLM {r['latency_ms']}ms]"
    if r["source"] == "llm":
        llm_calls += 1
    print(f"{label} {r['query']}")
    print(f"  {r['response'][:80]}...\n")

stats = result["cache_stats"]
print(f"Hit rate:      {stats['hit_rate'] * 100:.0f}%")
print(f"LLM calls:     {llm_calls} of {len(QUERIES)} queries")
print(f"LLM calls saved: {len(QUERIES) - llm_calls}")
print(f"Full stats:    {json.dumps(stats, indent=2)}")